### Notebook for processing the validation of the Futhark results

In [85]:
import ast
import re
import numpy as np
import pandas as pd
from pathlib import Path

##### Functions for processing Matlab and Futhark Prices

In [86]:
FUT_TYPE_SUFFIX = re.compile(r'(f16|f32|f64|i8|i16|i32|i64|u8|u16|u32|u64)\b')

def parse_futhark_prices(path):
    """Parse the price matrix from line 1 of a Futhark validate_solve .val file.

    The first line is a Futhark literal of shape [c][Ax], e.g.
    `[[200.0f64, 169.6f64, ...], [260.0f64, ...], ...]`.
    Returns a numpy.ndarray of shape (c, Ax) with dtype float64.
    """
    with open(path) as f:
        first_line = f.readline()
    cleaned = FUT_TYPE_SUFFIX.sub('', first_line)
    nested = ast.literal_eval(cleaned)
    return np.asarray(nested, dtype=np.float64)

def parse_matlab_prices(path):
    """Parse the price matrix from a MATLAB validate_run_illustrations_test .dat file.

    The file is a CSV (one row per car type, columns = ages 0..Ax-1).
    Returns a numpy.ndarray of shape (c, Ax) with dtype float64.
    """
    return np.loadtxt(path, delimiter=',', dtype=np.float64)

In [87]:
parse_futhark_prices('validation_files/run_equilibrium-local-validate_solve-2-3-25-5-0-M.val')

array([[200.        , 169.2948619 , 141.47740163, 117.0506919 ,
         95.96637377,  77.96530383,  62.62372555,  49.514447  ,
         38.31752597,  28.83231542,  20.94596407,  14.59768623,
          9.74911362,   6.33721219,   4.18410726,   2.96706019,
          2.33557   ,   2.0274529 ,   1.88412023,   1.81988041,
          1.79051281,   1.7704009 ,   1.73140671,   1.5988312 ,
          1.06290795],
       [260.        , 225.25506523, 193.02633663, 163.79794474,
        137.68705753, 114.73608295,  94.842273  ,  77.72158179,
         62.98567507,  50.27381226,  39.32270211,  29.96686878,
         22.11232877,  15.70826562,  10.72367414,   7.11425982,
          4.74873215,   3.35688607,   2.60697298,   2.22655565,
          2.03958087,   1.94245189,   1.8640367 ,   1.70520958,
          1.1400604 ],
       [260.        , 225.25506523, 193.02633663, 163.79794474,
        137.68705753, 114.73608295,  94.842273  ,  77.72158179,
         62.98567507,  50.27381226,  39.32270211,  29.9668

In [88]:
parse_matlab_prices('matlab_results_for_validation/matlab-validate_solve-2-3-25-5-0.dat')

array([[200.        , 169.29486084, 141.4773999 , 117.05068984,
         95.96637165,  77.96530179,  62.62372362,  49.5144452 ,
         38.3175243 ,  28.83231391,  20.94596275,  14.59768512,
          9.7491127 ,   6.33721146,   4.18410672,   2.96705985,
          2.3355698 ,   2.02745279,   1.88412018,   1.81988038,
          1.7905128 ,   1.7704009 ,   1.73140671,   1.5988312 ,
          1.06290794],
       [260.        , 225.25506359, 193.0263338 , 163.79794114,
        137.68705352, 114.73607879,  94.84226882,  77.72157762,
         62.98567095,  50.27380827,  39.32269838,  29.96686546,
         22.11232603,  15.70826358,  10.72367279,   7.11425896,
          4.7487316 ,   3.35688573,   2.60697279,   2.22655554,
          2.03958082,   1.94245186,   1.86403669,   1.70520957,
          1.14006039],
       [260.        , 225.25506359, 193.0263338 , 163.79794114,
        137.68705352, 114.73607879,  94.84226882,  77.72157762,
         62.98567095,  50.27380827,  39.32269838,  29.9668

##### Functions for comparing Futhark and Matlab

In [89]:
_FUT_NAME_RE = re.compile(r'validate_solve-(\d+)-(\d+)-(\d+)-(\d+)-(\d+)-[A-Z]\.val$')

def compare_validation(futhark_name, rtol=1e-4, atol=1e-6,
                         val_dir='validation_files',
                         mat_dir='matlab_results_for_validation'):
      """Compare prices in a Futhark .val file against the matching MATLAB .dat file.

      Pass/fail uses numpy.allclose semantics:
          |fut - mat| <= atol + rtol * |mat|

      Parameters
      ----------
      futhark_name : str
          Filename inside `val_dir`, e.g.
          'run_equilibrium-local-validate_solve-2-7-25-5-0-C.val'.
      rtol : float, default 1e-4
          Relative tolerance (dominates for large prices).
      atol : float, default 1e-6
          Absolute tolerance floor (dominates near scrap, where prices ~ 0).
      val_dir, mat_dir : str or Path
          Directories holding the Futhark and MATLAB validation files.

      Returns
      -------
      None
          If no MATLAB file matches the parameter tuple in `futhark_name`.
      dict
          Keys:
            'within_tol'   bool   passes np.allclose(fut, mat, rtol, atol)
            'max_abs_diff' float  max |fut - mat|
            'max_rel_diff' float  max |fut - mat| / max(|mat|, atol)
            'matlab_path'  Path
      """
      m = _FUT_NAME_RE.search(futhark_name)
      if m is None:
          raise ValueError(f"Could not extract parameter tuple from: {futhark_name}")
      n, c, abar, acc0, trans = m.groups()

      matlab_name = f"matlab-validate_solve-{n}-{c}-{abar}-{acc0}-{trans}.dat"
      matlab_path = Path(mat_dir) / matlab_name
      if not matlab_path.exists():
          return None

      fut = parse_futhark_prices(Path(val_dir) / futhark_name)
      mat = parse_matlab_prices(matlab_path)

      abs_diff = np.abs(fut - mat)
      max_abs_diff = float(np.max(abs_diff))
      max_rel_diff = float(np.max(abs_diff / np.maximum(np.abs(mat), atol)))
      return {
          'within_tol':   bool(np.allclose(fut, mat, rtol=rtol, atol=atol)),
          'max_abs_diff': max_abs_diff,
          'max_rel_diff': max_rel_diff,
          'matlab_path':  matlab_path,
      }

##### Futhark Functions for parsing and summarizing validation files

In [90]:
def parse_futhark_validation(path):
      """Parse all fields from a Futhark validate_solve .val file.

      Returns a dict with:
        'prices'        ndarray  shape (c, Ax), float64
        'max_abs_ed'    float
        'stat_res'      float
        'norm_err'      float
        'min_q'         float
        'iter'          int      Newton outer iterations
        'conv'          bool     convergence flag
        'sa_iters_tot'  list[int]  per-household SA iteration totals
        'nk_iters_tot'  list[int]  per-household NK iteration totals
        'rtrips_tot'    list[int]  per-household round-trip totals
      """
      with open(path) as f:
          lines = f.read().splitlines()

      def strip(s):
          return FUT_TYPE_SUFFIX.sub('', s)

      return {
          'prices':       np.asarray(ast.literal_eval(strip(lines[0])), dtype=np.float64),
          'max_abs_ed':   float(ast.literal_eval(strip(lines[1]))),
          'stat_res':     float(ast.literal_eval(strip(lines[2]))),
          'norm_err':     float(ast.literal_eval(strip(lines[3]))),
          'min_q':        float(ast.literal_eval(strip(lines[4]))),
          'iter':         int(ast.literal_eval(strip(lines[5]))),
          'conv':         lines[6].strip() == 'true',
          'sa_iters_tot': list(ast.literal_eval(strip(lines[7]))),
          'nk_iters_tot': list(ast.literal_eval(strip(lines[8]))),
          'rtrips_tot':   list(ast.literal_eval(strip(lines[9]))),
      }

In [91]:
def summarize_futhark_run(futhark_name, rtol=1e-4, atol=1e-6,
                            val_dir='validation_files',
                            mat_dir='matlab_results_for_validation'):
      """All Futhark validation fields (except the price matrix) plus the
      MATLAB comparison verdict.

      See parse_futhark_validation for the remaining field names. Adds:
        'within_tol'    bool or None  (None if no matching MATLAB file)
        'max_abs_diff'  float or None
        'max_rel_diff'  float or None
        'matlab_path'   Path  or None
      """
      fields = parse_futhark_validation(Path(val_dir) / futhark_name)
      del fields['prices']
      cmp = compare_validation(futhark_name, rtol=rtol, atol=atol,
                               val_dir=val_dir, mat_dir=mat_dir)
      if cmp is None:
          fields.update(within_tol=None, max_abs_diff=None,
                        max_rel_diff=None, matlab_path=None)
      else:
          fields.update(cmp)
      return fields

##### Summarize all Futhark Validation Files

In [92]:
_FUT_FILE_RE = re.compile(
      r'^(?P<run_equi>[^-]+)-(?P<val_name>.+?)-validate_solve-'
      r'(?P<n>\d+)-(?P<c>\d+)-(?P<abar>\d+)-(?P<acc0>\d+)-(?P<trans>\d+)-'
      r'(?P<backend>[A-Z])\.val$'
  )

def summarize_all_futhark(rtol=1e-4, atol=1e-6,
                            val_dir='validation_files',
                            mat_dir='matlab_results_for_validation'):
      """Summarize every .val file in `val_dir`.

      Each row is a dict carrying:
        filename       str       the .val filename
        run_equi       str       e.g. 'run_equilibrium' / 'run_equilibrium_man'
        val_name       str       VAL_NAME tag, e.g. 'local' / 'default'
        n, c, abar, acc0, trans  int   parameter tuple
        backend        str       'C', 'M', 'O', 'U'
      plus everything `summarize_futhark_run` returns (prices, max_abs_ed,
      stat_res, norm_err, min_q, iter, conv, sa_iters_tot, nk_iters_tot,
      rtrips_tot, within_tol, max_abs_diff, max_rel_diff, matlab_path).

      Files whose names don't match the validate_solve pattern are skipped.
      """
      val_dir = Path(val_dir)
      int_keys = {'n', 'c', 'abar', 'acc0', 'trans'}
      rows = []
      for path in sorted(val_dir.glob('*.val')):
          m = _FUT_FILE_RE.match(path.name)
          if m is None:
              continue
          meta = {k: int(v) if k in int_keys else v
                  for k, v in m.groupdict().items()}
          meta['filename'] = path.name
          summary = summarize_futhark_run(path.name, rtol=rtol, atol=atol,
                                          val_dir=val_dir, mat_dir=mat_dir)
          rows.append({**meta, **summary})
      return rows

In [93]:
rows = summarize_all_futhark()
print(rows[0])  # Print the first row, adjust index as needed

{'run_equi': 'run_equilibrium', 'val_name': '3_local_new', 'n': 2, 'c': 1, 'abar': 25, 'acc0': 5, 'trans': 0, 'backend': 'C', 'filename': 'run_equilibrium-3_local_new-validate_solve-2-1-25-5-0-C.val', 'max_abs_ed': 1.77606e-10, 'stat_res': 3e-15, 'norm_err': 0.0, 'min_q': 0.00047250047593, 'iter': 7, 'conv': True, 'sa_iters_tot': [30, 33], 'nk_iters_tot': [12, 7], 'rtrips_tot': [7, 7], 'within_tol': True, 'max_abs_diff': 2.064203300733425e-06, 'max_rel_diff': 1.0413759811939476e-07, 'matlab_path': WindowsPath('matlab_results_for_validation/matlab-validate_solve-2-1-25-5-0.dat')}


##### Benchmark parsing

In [94]:
def parse_matlab_benchmark(path):
    """Parse matlab_eqb_<variant>.dat. Columns: n c abar acc0 trans mean stdev se."""
    arr = np.loadtxt(path)
    if arr.ndim == 1:
        arr = arr[None, :]
    cols = ['n', 'c', 'abar', 'acc0', 'trans', 'mean', 'stdev', 'se']
    return [dict(zip(cols, row)) for row in arr]

def parse_futhark_benchmark(path):
    """Parse a Futhark bench_solve_<variant>.dat (or saved copy).
    Columns: n c abar acc0 trans backend mean stdev se."""
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 9:
                continue
            rows.append({
                'n':       int(parts[0]),  'c':       int(parts[1]),
                'abar':    int(parts[2]),  'acc0':    int(parts[3]),
                'trans':   int(parts[4]),  'backend': parts[5],
                'mean':    float(parts[6]),
                'stdev':   float(parts[7]),
                'se':      float(parts[8]),
            })
    return rows

In [95]:
_SAVED_BENCH_RE = re.compile(
      r'^(?P<run_equi>[^-]+)-(?P<val_name>.+?)-bench(?P<runs>\d+)-'
      r'(?P<variant>cars_longest|cars_longer|cars_long|cars|households|age)\.dat$'
  )

def bench_time_table(variant, saved_dir='saved_futhark_benchmarks'):
    """Long-format DataFrame of bench mean & stdev for the given variant.
    One row per (source, run_equi, val_name, backend, parameter tuple)."""
    rows = []

    mat_path = Path(f'matlab_eqb_{variant}.dat')
    if mat_path.exists():
        for r in parse_matlab_benchmark(mat_path):
            rows.append({
                'source': 'matlab', 'run_equi': '-', 'val_name': '-',
                'backend': '-',     'runs': '-',
                'n': int(r['n']), 'c': int(r['c']), 'abar': int(r['abar']),
                'mean': r['mean'], 'stdev': r['stdev'],
            })

    for path in sorted(Path(saved_dir).glob(f'*-{variant}.dat')):
        m = _SAVED_BENCH_RE.match(path.name)
        if m is None:
            continue
        meta = m.groupdict()
        for r in parse_futhark_benchmark(path):
            rows.append({
                'source':   'futhark',
                'run_equi': meta['run_equi'],
                'val_name': meta['val_name'],
                'backend':  r['backend'],
                'runs':     int(meta['runs']),
                'n': r['n'], 'c': r['c'], 'abar': r['abar'],
                'mean': r['mean'], 'stdev': r['stdev'],
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        sort_key = {'cars': 'c', 'households': 'n', 'age': 'abar', 'cars_long': 'c', 'cars_longer': 'c', 'cars_longest': 'c'}[variant]
        df = df.sort_values([sort_key, 'source', 'run_equi', 'val_name', 'backend'])
    return df.reset_index(drop=True)

##### Comparison Table for Validation

In [96]:
def comparison_table(variant, rtol=1e-4, atol=1e-6,
                       val_dir='validation_files',
                       mat_dir='matlab_results_for_validation'):
      """DataFrame summarising every .val file that belongs to this variant.

      Each row: one (run_equi, val_name, backend, parameter tuple) combination,
      with columns including within_tol, max_abs_diff, max_rel_diff, and the
      parsed Futhark fields (iter, conv, sa_iters_tot, nk_iters_tot, etc.).

      Note: cars_long is intentionally folded into 'cars' here -- the 'cars'
      validation table covers the whole (n=2, abar=25) plane, including the
      extended c values (17, 21). cars_long is only a separate variant for the
      bench-time tables, which read variant-specific .dat files.
      """
      rows = summarize_all_futhark(rtol=rtol, atol=atol,
                                   val_dir=val_dir, mat_dir=mat_dir)
      if variant == 'cars':
          keep = lambda r: r['n'] == 2 and r['abar'] == 25
          sort_key = 'c'
      elif variant == 'households':
          keep = lambda r: r['c'] == 7 and r['abar'] == 25
          sort_key = 'n'
      elif variant == 'age':
          keep = lambda r: r['n'] == 2 and r['c'] == 7
          sort_key = 'abar'
      else:
          raise ValueError(f"Unknown variant: {variant!r}")

      df = pd.DataFrame([r for r in rows if keep(r)])
      if not df.empty:
          df = (df.drop(columns=['matlab_path'], errors='ignore')
                  .sort_values(['run_equi', 'val_name', 'backend', sort_key])
                  .reset_index(drop=True))
      return df

##### Tables

In [97]:
for v in ['cars', 'households', 'age', 'cars_long']:
    print(f"\n=== {v}: bench times ===")
    display(bench_time_table(v))

for v in ['cars', 'households', 'age']:
    print(f"\n=== {v}: MATLAB comparison + Futhark diagnostics ===")
    display(comparison_table(v))


=== cars: bench times ===


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
0,futhark,run_equilibrium,3_local_new,C,3,2,1,25,0.021070,0.001137
1,futhark,run_equilibrium,3_local_new,M,3,2,1,25,0.115907,0.040948
2,futhark,run_equilibrium,default,C,5,2,1,25,0.071775,0.000074
3,futhark,run_equilibrium,default,M,5,2,1,25,0.070213,0.000340
4,futhark,run_equilibrium,default,U,5,2,1,25,0.166022,0.108779
5,futhark,run_equilibrium,futhark01,C,10,2,1,25,0.090386,0.000233
6,futhark,run_equilibrium,futhark01,C,3,2,1,25,0.078432,0.000819
7,futhark,run_equilibrium,futhark01,M,10,2,1,25,0.075278,0.004125
8,futhark,run_equilibrium,futhark01,M,3,2,1,25,0.071403,0.005256
9,futhark,run_equilibrium,futhark01,U,10,2,1,25,0.685231,0.149442



=== households: bench times ===


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
0,futhark,run_equilibrium,futhark01,C,3,2,7,25,12.046656,0.480690
1,futhark,run_equilibrium,futhark01,M,3,2,7,25,1.865896,0.447884
2,futhark,run_equilibrium,futhark01,U,3,2,7,25,1.564589,0.122438
3,futhark,run_equilibrium,local,C,3,2,7,25,3.311001,0.021470
4,futhark,run_equilibrium,local,M,3,2,7,25,0.987857,0.025682
5,futhark,run_equilibrium_dp_solve,local,C,3,2,7,25,12.550822,0.391622
6,futhark,run_equilibrium_dp_solve,local,M,3,2,7,25,2.488624,0.013574
7,futhark,run_equilibrium_full_ad,local,C,3,2,7,25,6.365089,0.054998
8,futhark,run_equilibrium_full_ad,local,M,3,2,7,25,1.655991,0.075962
9,futhark,run_equilibrium_man,futhark01,C,3,2,7,25,1.795280,0.011455



=== age: bench times ===


...

,run_equi,val_name,n,c,abar,acc0,trans,backend,filename,max_abs_ed,stat_res,norm_err,min_q,iter,conv,sa_iters_tot,nk_iters_tot,rtrips_tot,within_tol,max_abs_diff,max_rel_diff
0,run_equilibrium,3_local_new,2,1,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-1...,1.776060e-10,3.000000e-15,0.000000e+00,4.725005e-04,7,True,"[30, 33]","[12, 7]","[7, 7]",True,0.000002,1.041376e-07
1,run_equilibrium,3_local_new,2,3,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-3...,1.500000e-14,5.300000e-14,0.000000e+00,2.429617e-04,8,True,"[32, 32]","[8, 8]","[8, 8]",True,0.000004,1.298640e-07
2,run_equilibrium,3_local_new,2,5,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-5...,1.199000e-12,3.700000e-14,1.000000e-15,1.645633e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,2.444766e-07
3,run_equilibrium,3_local_new,2,7,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-7...,9.000000e-14,2.500000e-13,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
4,run_equilibrium,3_local_new,2,9,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-9...,1.601310e-10,1.190000e-13,1.000000e-15,1.002891e-04,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.857835e-07
5,run_equilibrium,3_local_new,2,11,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-1...,1.488700e-11,3.640000e-13,1.000000e-15,8.393742e-05,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.817334e-07
6,run_equilibrium,3_local_new,2,13,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-1...,3.034000e-12,2.380000e-13,1.000000e-15,7.217770e-05,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.614682e-07
7,run_equilibrium,3_local_new,2,1,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-1...,1.776060e-10,6.000000e-15,0.000000e+00,4.725005e-04,7,True,"[30, 33]","[12, 7]","[7, 7]",True,0.000002,1.041376e-07
8,run_equilibrium,3_local_new,2,3,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-3...,1.500000e-14,5.300000e-14,0.000000e+00,2.429617e-04,8,True,"[32, 32]","[8, 8]","[8, 8]",True,0.000004,1.298640e-07
9,run_equilibrium,3_local_new,2,5,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-5...,1.199000e-12,3.700000e-14,1.000000e-15,1.645633e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,2.444766e-07



=== households: MATLAB comparison + Futhark diagnostics ===


,run_equi,val_name,n,c,abar,acc0,trans,backend,filename,max_abs_ed,stat_res,norm_err,min_q,iter,conv,sa_iters_tot,nk_iters_tot,rtrips_tot,within_tol,max_abs_diff,max_rel_diff
0,run_equilibrium,3_local_new,2,7,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-7...,9.000000e-14,2.500000e-13,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
1,run_equilibrium,3_local_new,2,7,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-7...,9.000000e-14,2.500000e-13,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
2,run_equilibrium,futhark01,2,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-2...,1.170000e-13,9.000000e-14,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635888e-07
3,run_equilibrium,futhark01,3,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-3-7-2...,6.000000e-15,3.600000e-14,0.000000e+00,0.000273,6,True,"[24, 24, 24]","[6, 6, 6]","[6, 6, 6]",True,0.000007,3.291834e-07
4,run_equilibrium,futhark01,4,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-4-7-2...,9.100000e-14,8.200000e-14,1.000000e-15,0.000384,6,True,"[24, 24, 24, 24]","[6, 6, 6, 6]","[6, 6, 6, 6]",True,0.000005,2.529969e-07
5,run_equilibrium,futhark01,5,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-5-7-2...,6.869810e-10,1.020000e-13,0.000000e+00,0.000457,5,True,"[20, 20, 20, 20, 20]","[5, 5, 5, 5, 5]","[5, 5, 5, 5, 5]",True,0.000006,2.735996e-07
6,run_equilibrium,futhark01,6,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-6-7-2...,5.541980e-10,2.800000e-14,1.000000e-15,0.000505,5,True,"[20, 20, 20, 20, 20, 20]","[5, 5, 5, 5, 5, 5]","[5, 5, 5, 5, 5, 5]",True,0.000006,2.805666e-07
7,run_equilibrium,futhark01,2,7,25,5,0,M,run_equilibrium-futhark01-validate_solve-2-7-2...,1.760000e-13,4.900000e-14,0.000000e+00,0.000125,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635900e-07
8,run_equilibrium,futhark01,3,7,25,5,0,M,run_equilibrium-futhark01-validate_solve-3-7-2...,6.000000e-15,3.600000e-14,0.000000e+00,0.000273,6,True,"[24, 24, 24]","[6, 6, 6]","[6, 6, 6]",True,0.000007,3.291834e-07
9,run_equilibrium,futhark01,4,7,25,5,0,M,run_equilibrium-futhark01-validate_solve-4-7-2...,9.100000e-14,8.200000e-14,0.000000e+00,0.000384,6,True,"[24, 24, 24, 24]","[6, 6, 6, 6]","[6, 6, 6, 6]",True,0.000005,2.529969e-07



=== age: MATLAB comparison + Futhark diagnostics ===


,run_equi,val_name,n,c,abar,acc0,trans,backend,filename,max_abs_ed,stat_res,norm_err,min_q,iter,conv,sa_iters_tot,nk_iters_tot,rtrips_tot,within_tol,max_abs_diff,max_rel_diff
0,run_equilibrium,3_local_new,2,7,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-7...,9.000000e-14,2.500000e-13,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
1,run_equilibrium,3_local_new,2,7,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-7...,9.000000e-14,2.500000e-13,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
2,run_equilibrium,futhark01,2,7,10,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-1...,2.167800e-11,6.900000e-14,1.000000e-15,9.404310e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000002,4.316708e-08
3,run_equilibrium,futhark01,2,7,15,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-1...,1.100000e-14,1.000000e-13,1.000000e-15,8.037581e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000005,1.926052e-07
4,run_equilibrium,futhark01,2,7,20,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-2...,8.179000e-12,9.200000e-14,0.000000e+00,1.622109e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.662682e-07
5,run_equilibrium,futhark01,2,7,25,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-2...,1.170000e-13,9.000000e-14,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635888e-07
6,run_equilibrium,futhark01,2,7,30,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-3...,1.500000e-13,1.460000e-13,1.000000e-15,1.013898e-05,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000007,2.794752e-07
7,run_equilibrium,futhark01,2,7,35,5,0,C,run_equilibrium-futhark01-validate_solve-2-7-3...,2.370000e-13,1.180000e-13,1.000000e-15,8.315462e-07,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000007,2.799875e-07
8,run_equilibrium,futhark01,2,7,10,5,0,M,run_equilibrium-futhark01-validate_solve-2-7-1...,2.167800e-11,6.900000e-14,1.000000e-15,9.404310e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000002,4.316708e-08
9,run_equilibrium,futhark01,2,7,15,5,0,M,run_equilibrium-futhark01-validate_solve-2-7-1...,1.100000e-14,1.000000e-13,1.000000e-15,8.037581e-03,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000005,1.926052e-07


In [98]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(bench_time_table("cars_long"))

for system in ["local", "futhark01"]:
    for run_equi in ["run_equilibrium", "run_equilibrium_man"]:
        df = bench_time_table("cars_long")
        df = df[
            ((df["source"] == "matlab") |
             ((df["source"] == "futhark") &
              (df["val_name"] == system) &
              (df["run_equi"] == run_equi)))
        ]
        display(df)

,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
0,futhark,run_equilibrium,futhark01,C,3,2,1,25,0.077338,0.000086
1,futhark,run_equilibrium,futhark01,M,3,2,1,25,0.065449,0.006283
2,futhark,run_equilibrium,futhark01,U,3,2,1,25,0.289159,0.000572
3,futhark,run_equilibrium,local,C,3,2,1,25,0.015431,0.000122
4,futhark,run_equilibrium,local,M,3,2,1,25,0.047325,0.014614
5,futhark,run_equilibrium_almost_full_ad,futhark01,C,3,2,1,25,0.073071,0.000068
6,futhark,run_equilibrium_almost_full_ad,futhark01,M,3,2,1,25,0.083028,0.007827
7,futhark,run_equilibrium_almost_full_ad,futhark01,U,3,2,1,25,0.285914,0.000693
8,futhark,run_equilibrium_almost_full_ad,local,C,3,2,1,25,0.014388,0.000287
9,futhark,run_equilibrium_almost_full_ad,local,M,3,2,1,25,0.038469,0.007098


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
3,futhark,run_equilibrium,local,C,3,2,1,25,0.015431,0.000122
4,futhark,run_equilibrium,local,M,3,2,1,25,0.047325,0.014614
21,matlab,-,-,-,-,2,1,25,0.105785,0.027910
25,futhark,run_equilibrium,local,C,3,2,5,25,1.270393,0.037300
26,futhark,run_equilibrium,local,M,3,2,5,25,0.308382,0.009547
43,matlab,-,-,-,-,2,5,25,1.754218,0.030317
47,futhark,run_equilibrium,local,C,3,2,9,25,5.712635,0.093886
48,futhark,run_equilibrium,local,M,3,2,9,25,1.842999,0.079794
65,matlab,-,-,-,-,2,9,25,10.211754,0.054219
69,futhark,run_equilibrium,local,C,3,2,13,25,18.838399,1.161052


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
17,futhark,run_equilibrium_man,local,C,3,2,1,25,0.005668,0.000408
18,futhark,run_equilibrium_man,local,M,3,2,1,25,0.060869,0.017337
21,matlab,-,-,-,-,2,1,25,0.105785,0.027910
39,futhark,run_equilibrium_man,local,C,3,2,5,25,0.271691,0.003122
40,futhark,run_equilibrium_man,local,M,3,2,5,25,0.217664,0.029567
43,matlab,-,-,-,-,2,5,25,1.754218,0.030317
61,futhark,run_equilibrium_man,local,C,3,2,9,25,1.354198,0.031164
62,futhark,run_equilibrium_man,local,M,3,2,9,25,0.935198,0.064591
65,matlab,-,-,-,-,2,9,25,10.211754,0.054219
83,futhark,run_equilibrium_man,local,C,3,2,13,25,3.870256,0.006960


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
0,futhark,run_equilibrium,futhark01,C,3,2,1,25,0.077338,0.000086
1,futhark,run_equilibrium,futhark01,M,3,2,1,25,0.065449,0.006283
2,futhark,run_equilibrium,futhark01,U,3,2,1,25,0.289159,0.000572
21,matlab,-,-,-,-,2,1,25,0.105785,0.027910
22,futhark,run_equilibrium,futhark01,C,3,2,5,25,3.934542,0.309512
23,futhark,run_equilibrium,futhark01,M,3,2,5,25,0.588528,0.044075
24,futhark,run_equilibrium,futhark01,U,3,2,5,25,1.312285,0.000909
43,matlab,-,-,-,-,2,5,25,1.754218,0.030317
44,futhark,run_equilibrium,futhark01,C,3,2,9,25,17.633968,0.075447
45,futhark,run_equilibrium,futhark01,M,3,2,9,25,1.371026,0.084310


,source,run_equi,val_name,backend,runs,n,c,abar,mean,stdev
14,futhark,run_equilibrium_man,futhark01,C,3,2,1,25,0.024319,0.000030
15,futhark,run_equilibrium_man,futhark01,M,3,2,1,25,0.058998,0.000799
16,futhark,run_equilibrium_man,futhark01,U,3,2,1,25,0.292506,0.000426
21,matlab,-,-,-,-,2,1,25,0.105785,0.027910
36,futhark,run_equilibrium_man,futhark01,C,3,2,5,25,0.781393,0.191018
37,futhark,run_equilibrium_man,futhark01,M,3,2,5,25,0.403040,0.016153
38,futhark,run_equilibrium_man,futhark01,U,3,2,5,25,2.378817,0.514461
43,matlab,-,-,-,-,2,5,25,1.754218,0.030317
58,futhark,run_equilibrium_man,futhark01,C,3,2,9,25,3.224055,0.008248
59,futhark,run_equilibrium_man,futhark01,M,3,2,9,25,1.272435,0.010355


##### LaTeX Table

In [99]:
def local_bench_table(variant, val_name='local'):
      """Wide table for `variant`: rows = varying parameter (c / n / abar),
      columns = (MATLAB / Futhark-C / Futhark-M) × (mean, stdev).
      Only Futhark rows whose val_name matches `val_name` are kept."""
      df = bench_time_table(variant)

      mask = (df['source'] == 'matlab') | (
          (df['source'] == 'futhark') & (df['val_name'] == val_name)
      )
      df = df[mask].copy()
      df['label'] = df.apply(
          lambda r: 'MATLAB' if r['source'] == 'matlab' else f"Futhark-{r['backend']}",
          axis=1,
      )

      var_col = {'cars': 'c', 'households': 'n', 'age': 'abar', 'cars_long': 'c'}[variant]
      pivot = df.pivot_table(index=var_col, columns='label',
                             values=['mean', 'stdev'], aggfunc='first')
      pivot = pivot.swaplevel(0, 1, axis=1)

      label_order = ['MATLAB', 'Futhark-C', 'Futhark-M']
      present = [l for l in label_order if l in pivot.columns.get_level_values(0).unique()]
      cols = [(lbl, stat) for lbl in present for stat in ['mean', 'stdev']]
      return pivot.reindex(columns=cols)

In [100]:
for v in ['cars', 'households', 'age', 'cars_long']:
      print(f"\n=== {v}: bench times (local) ===")
      tbl = local_bench_table(v)
      display(tbl)
      print(tbl.to_latex(float_format='%.4f',
                         caption=f'Equilibrium benchmark times ({v})',
                         label=f'tab:bench-{v}'))


=== cars: bench times (local) ===


label     MATLAB            Futhark-C           Futhark-M          
            mean     stdev       mean     stdev      mean     stdev
c                                                                  
1       0.068386  0.020576   0.015431  0.000122  0.047325  0.014614
3       0.485418  0.086544   0.330239  0.000843  0.132433  0.008728
5       1.939685  0.108326   1.270393  0.037300  0.308382  0.009547
7       7.587036  0.504120   3.311001  0.021470  0.987857  0.025682
9      11.064450  0.675593   5.712635  0.093886  1.842999  0.079794
11     19.586699  0.637146  10.658316  0.191747  3.224413  0.022561
13     30.973843  0.538623  18.838399  1.161052  5.590788  0.113585

\begin{table}
\caption{Equilibrium benchmark times (cars)}
\label{tab:bench-cars}
\begin{tabular}{lrrrrrr}
\toprule
label & \multicolumn{2}{r}{MATLAB} & \multicolumn{2}{r}{Futhark-C} & \multicolumn{2}{r}{Futhark-M} \\
 & mean & stdev & mean & stdev & mean & stdev \\
c &  &  &  &  &  &  \\
\midrule
1 & 0.0684 & 0.0206 & 0.0154 & 0.0001 & 0.0473 & 0.0146 \\
3 & 0.4854 & 0.0865 & 0.3302 & 0.0008 & 0.1324 & 0.0087 \\
5 & 1.9397 & 0.1083 & 1.2704 & 0.0373 & 0.3084 & 0.0095 \\
7 & 7.5870 & 0.5041 & 3.3110 & 0.0215 & 0.9879 & 0.0257 \\
9 & 11.0645 & 0.6756 & 5.7126 & 0.0939 & 1.8430 & 0.0798 \\
11 & 19.5867 & 0.6371 & 10.6583 & 0.1917 & 3.2244 & 0.0226 \\
13 & 30.9738 & 0.5386 & 18.8384 & 1.1611 & 5.5908 & 0.1136 \\
\bottomrule
\end{tabular}
\end{table}


=== households: bench times (local) ===


label     MATLAB           Futhark-C           Futhark-M          
            mean     stdev      mean     stdev      mean     stdev
n                                                                 
2       7.020355  0.190719  3.311001  0.021470  0.987857  0.025682
3       6.707738  0.021294  4.233363  0.126168  1.175821  0.058068
4       8.168758  0.021005  5.326726  0.074081  1.498123  0.075818
5      11.204162  0.043375  6.525457  0.033199  1.755546  0.090920
6      11.174867  0.044354  7.827898  0.052284  2.442221  0.205999

\begin{table}
\caption{Equilibrium benchmark times (households)}
\label{tab:bench-households}
\begin{tabular}{lrrrrrr}
\toprule
label & \multicolumn{2}{r}{MATLAB} & \multicolumn{2}{r}{Futhark-C} & \multicolumn{2}{r}{Futhark-M} \\
 & mean & stdev & mean & stdev & mean & stdev \\
n &  &  &  &  &  &  \\
\midrule
2 & 7.0204 & 0.1907 & 3.3110 & 0.0215 & 0.9879 & 0.0257 \\
3 & 6.7077 & 0.0213 & 4.2334 & 0.1262 & 1.1758 & 0.0581 \\
4 & 8.1688 & 0.0210 & 5.3267 & 0.0741 & 1.4981 & 0.0758 \\
5 & 11.2042 & 0.0434 & 6.5255 & 0.0332 & 1.7555 & 0.0909 \\
6 & 11.1749 & 0.0444 & 7.8279 & 0.0523 & 2.4422 & 0.2060 \\
\bottomrule
\end{tabular}
\end{table}


=== age: bench times (local) ===


label     MATLAB           Futhark-C           Futhark-M          
            mean     stdev      mean     stdev      mean     stdev
abar                                                              
10      0.681014  0.037932  0.275749  0.007175  0.105716  0.014869
15      1.110800  0.038380  0.668816  0.003574  0.327886  0.078854
20      2.574031  0.156272  1.439351  0.025735  0.397220  0.011858
25      7.087320  0.248254  3.311001  0.021470  0.987857  0.025682
30     11.409867  0.942833  5.562934  0.027941  1.846307  0.113170
35     19.765368  0.748713  8.920049  0.068249  2.925500  0.087916

\begin{table}
\caption{Equilibrium benchmark times (age)}
\label{tab:bench-age}
\begin{tabular}{lrrrrrr}
\toprule
label & \multicolumn{2}{r}{MATLAB} & \multicolumn{2}{r}{Futhark-C} & \multicolumn{2}{r}{Futhark-M} \\
 & mean & stdev & mean & stdev & mean & stdev \\
abar &  &  &  &  &  &  \\
\midrule
10 & 0.6810 & 0.0379 & 0.2757 & 0.0072 & 0.1057 & 0.0149 \\
15 & 1.1108 & 0.0384 & 0.6688 & 0.0036 & 0.3279 & 0.0789 \\
20 & 2.5740 & 0.1563 & 1.4394 & 0.0257 & 0.3972 & 0.0119 \\
25 & 7.0873 & 0.2483 & 3.3110 & 0.0215 & 0.9879 & 0.0257 \\
30 & 11.4099 & 0.9428 & 5.5629 & 0.0279 & 1.8463 & 0.1132 \\
35 & 19.7654 & 0.7487 & 8.9200 & 0.0682 & 2.9255 & 0.0879 \\
\bottomrule
\end{tabular}
\end{table}


=== cars_long: bench times (local) ===


label      MATLAB             Futhark-C            Futhark-M          
             mean      stdev       mean     stdev       mean     stdev
c                                                                     
1        0.105785   0.027910   0.015431  0.000122   0.047325  0.014614
5        1.754218   0.030317   1.270393  0.037300   0.308382  0.009547
9       10.211754   0.054219   5.712635  0.093886   1.842999  0.079794
13      33.853231   3.864335  18.838399  1.161052   5.590788  0.113585
17     118.626387   9.634622  42.287842  0.384940  13.801242  0.162382
21     407.037305  36.981903  79.799575  0.705347  27.488477  1.779575

\begin{table}
\caption{Equilibrium benchmark times (cars_long)}
\label{tab:bench-cars_long}
\begin{tabular}{lrrrrrr}
\toprule
label & \multicolumn{2}{r}{MATLAB} & \multicolumn{2}{r}{Futhark-C} & \multicolumn{2}{r}{Futhark-M} \\
 & mean & stdev & mean & stdev & mean & stdev \\
c &  &  &  &  &  &  \\
\midrule
1 & 0.1058 & 0.0279 & 0.0154 & 0.0001 & 0.0473 & 0.0146 \\
5 & 1.7542 & 0.0303 & 1.2704 & 0.0373 & 0.3084 & 0.0095 \\
9 & 10.2118 & 0.0542 & 5.7126 & 0.0939 & 1.8430 & 0.0798 \\
13 & 33.8532 & 3.8643 & 18.8384 & 1.1611 & 5.5908 & 0.1136 \\
17 & 118.6264 & 9.6346 & 42.2878 & 0.3849 & 13.8012 & 0.1624 \\
21 & 407.0373 & 36.9819 & 79.7996 & 0.7053 & 27.4885 & 1.7796 \\
\bottomrule
\end{tabular}
\end{table}



##### Validation Tables

In [101]:
def _pivot_validation(variant, rename, val_name='local'):
      """Pivot validation rows for `variant`, keeping only the metrics in `rename`."""
      rows = comparison_table(variant)
      rows = rows[rows['val_name'] == val_name].copy()
      if rows.empty:
          return pd.DataFrame()

      metrics = list(rename)

      # Compact list-valued counters: [30, 30] -> '30';  [30, 33] -> '30,33'
      def compact(lst):
          return str(lst[0]) if len(set(lst)) == 1 else ','.join(map(str, lst))
      for col in {'sa_iters_tot', 'nk_iters_tot', 'rtrips_tot'} & set(metrics):
          rows[col] = rows[col].apply(compact)

      var_col = {'cars': 'c', 'households': 'n', 'age': 'abar'}[variant]
      rows['label'] = rows['run_equi'] + '-' + rows['backend']

      indexed = rows.set_index([var_col, 'label'])[metrics]
      pivot = indexed.unstack('label').rename(columns=rename, level=0)
      pivot = pivot.swaplevel(0, 1, axis=1)

      label_order = sorted(rows['label'].unique())
      cols = [(lbl, rename[m]) for lbl in label_order for m in metrics]
      return pivot.reindex(columns=cols)


def local_validation_summary(variant, val_name='local'):
    """Newton / Conv / Match per (run_equi, backend)."""
    return _pivot_validation(variant,
        {'iter': 'Newton', 'conv': 'Conv', 'within_tol': 'Match'}, val_name)


def local_validation_iters(variant, val_name='local'):
    """SA / NK / RT per (run_equi, backend)."""
    return _pivot_validation(variant,
        {'sa_iters_tot': 'SA', 'nk_iters_tot': 'NK', 'rtrips_tot': 'RT'}, val_name)

In [102]:
for v in ['cars', 'households', 'age']:
      print(f"\n=== {v}: convergence summary (local) ===")
      tbl = local_validation_summary(v)
      display(tbl)
      print(tbl.to_latex(caption=f'Convergence summary ({v})',
                         label=f'tab:val-summary-{v}'))

      print(f"\n=== {v}: iteration counts (local) ===")
      tbl = local_validation_iters(v)
      display(tbl)
      print(tbl.to_latex(caption=f'Iteration counts ({v})',
                         label=f'tab:val-iters-{v}'))


=== cars: convergence summary (local) ===


label run_equilibrium-C             run_equilibrium-M              \
                 Newton  Conv Match            Newton  Conv Match   
c                                                                   
1                   7.0  True  True               7.0  True  True   
3                   7.0  True  True               7.0  True  True   
5                   6.0  True  True               6.0  True  True   
7                   6.0  True  True               6.0  True  True   
9                   5.0  True  True               5.0  True  True   
11                  5.0  True  True               5.0  True  True   
13                  5.0  True  True               5.0  True  True   
17                  5.0  True  True               5.0  True  True   
21                  5.0  True  True               5.0  True  True   

label run_equilibrium_almost_full_ad-C              \
                                Newton  Conv Match   
c                                                    
1                                  7.0  True  True   
3                                  NaN   NaN   NaN   
5                                  6.0  True  True   
7                                  NaN   NaN   NaN   
9                                  5.0  True  True   
11                                 NaN   NaN   NaN   
13                                 5.0  True  True   
17                                 5.0  True  True   
21                                 5.0  True  True   

label run_equilibrium_almost_full_ad-M             run_equilibrium_dp_solve-C  \
                                Newton  Conv Match                     Newton   
c                                                                               
1                                  7.0  True  True                        7.0   
3                                  NaN   NaN   NaN                        7.0   
5                                  6.0  True  True                        6.0   
7                                  NaN   NaN   NaN                        6.0   
9                                  5.0  True  True                        6.0   
11                                 NaN   NaN   NaN                        5.0   
13                                 5.0  True  True                        5.0   
17                                 5.0  True  True                        NaN   
21                                 5.0  True  True                        NaN   

label             run_equilibrium_dp_solve-M              \
       Conv Match                     Newton  Conv Match   
c                                                          
1      True  True                        7.0  True  True   
3      True  True                        7.0  True  True   
5      True  True                        6.0  True  True   
7      True  True                        6.0  True  True   
9      True  True                        6.0  True  True   
11     True  True                        5.0  True  True   
13     True  True                        5.0  True  True   
17      NaN   NaN                        NaN   NaN   NaN   
21      NaN   NaN                        NaN   NaN   NaN   

label run_equilibrium_full_ad-C             run_equilibrium_full_ad-M        \
                         Newton  Conv Match                    Newton  Conv   
c                                                                             
1                           7.0  True  True                       7.0  True   
3                           8.0  True  True                       8.0  True   
5                           7.0  True  True                       7.0  True   
7                           7.0  True  True                       7.0  True   
9                           6.0  True  True                       6.0  True   
11                          6.0  True  True                       6.0  True   
13                          6.0  True  True                       6.0  True   
17                          5.0  True 

\begin{table}
\caption{Convergence summary (cars)}
\label{tab:val-summary-cars}
\begin{tabular}{lrllrllrllrllrllrllrllrllrllrllrllrllrllrllrllrll}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_almost_full_ad-C} & \multicolumn{3}{r}{run_equilibrium_almost_full_ad-M} & \multicolumn{3}{r}{run_equilibrium_dp_solve-C} & \multicolumn{3}{r}{run_equilibrium_dp_solve-M} & \multicolumn{3}{r}{run_equilibrium_full_ad-C} & \multicolumn{3}{r}{run_equilibrium_full_ad-M} & \multicolumn{3}{r}{run_equilibrium_full_ad_alt-C} & \multicolumn{3}{r}{run_equilibrium_full_ad_alt-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} & \multicolumn{3}{r}{run_equilibrium_man_sa-C} & \multicolumn{3}{r}{run_equilibrium_man_sa-M} & \multicolumn{3}{r}{run_equilibrium_sa-C} & \multicolumn{3}{r}{run_equilibrium_sa-M} \\
 & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newt

label run_equilibrium-C          run_equilibrium-M           \
                     SA    NK RT                SA    NK RT   
c                                                             
1                 30,33  12,7  7             30,33  12,7  7   
3                    28     7  7                28     7  7   
5                    24     6  6                24     6  6   
7                    24     6  6                24     6  6   
9                    20     5  5                20     5  5   
11                   20     5  5                20     5  5   
13                   20     5  5                20     5  5   
17                   20     5  5                20     5  5   
21                   20     5  5                20     5  5   

label run_equilibrium_almost_full_ad-C             \
                                    SA    NK   RT   
c                                                   
1                                30,33  12,7    7   
3                                  NaN   NaN  NaN   
5                                   24     6    6   
7                                  NaN   NaN  NaN   
9                                   20     5    5   
11                                 NaN   NaN  NaN   
13                                  20     5    5   
17                                  20     5    5   
21                                  20     5    5   

label run_equilibrium_almost_full_ad-M            run_equilibrium_dp_solve-C  \
                                    SA    NK   RT                         SA   
c                                                                              
1                                30,33  12,7    7                         -1   
3                                  NaN   NaN  NaN                         -1   
5                                   24     6    6                         -1   
7                                  NaN   NaN  NaN                         -1   
9                                   20     5    5                         -1   
11                                 NaN   NaN  NaN                         -1   
13                                  20     5    5                         -1   
17                                  20     5    5                        NaN   
21                                  20     5    5                        NaN   

label           run_equilibrium_dp_solve-M            \
        NK   RT                         SA   NK   RT   
c                                                      
1       -1   -1                         -1   -1   -1   
3       -1   -1                         -1   -1   -1   
5       -1   -1                         -1   -1   -1   
7       -1   -1                         -1   -1   -1   
9       -1   -1                         -1   -1   -1   
11      -1   -1                         -1   -1   -1   
13      -1   -1                         -1   -1   -1   
17     NaN  NaN                        NaN  NaN  NaN   
21     NaN  NaN                        NaN  NaN  NaN   

label run_equilibrium_full_ad-C          run_equilibrium_full_ad-M           \
                             SA    NK RT                        SA    NK RT   
c                                                                             
1                         30,33  12,7  7                     30,33  12,7  7   
3                            32     8  8                        32     8  8   
5                            28     7  7                        28     7  7   
7                            28     7  7                        28     7  7   
9                            24     6  6                        24     6  6   
11                           24     6  6                        24     6  6   
13                           24     6  6                        24     6  6   
17                           20     5  5                        20     5  5   
21                           20     5  5                        20     5  5   

label run_equilibrium_f

\begin{table}
\caption{Iteration counts (cars)}
\label{tab:val-iters-cars}
\begin{tabular}{lllllllllllllllllllllllllllllllllllllllllllllllll}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_almost_full_ad-C} & \multicolumn{3}{r}{run_equilibrium_almost_full_ad-M} & \multicolumn{3}{r}{run_equilibrium_dp_solve-C} & \multicolumn{3}{r}{run_equilibrium_dp_solve-M} & \multicolumn{3}{r}{run_equilibrium_full_ad-C} & \multicolumn{3}{r}{run_equilibrium_full_ad-M} & \multicolumn{3}{r}{run_equilibrium_full_ad_alt-C} & \multicolumn{3}{r}{run_equilibrium_full_ad_alt-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} & \multicolumn{3}{r}{run_equilibrium_man_sa-C} & \multicolumn{3}{r}{run_equilibrium_man_sa-M} & \multicolumn{3}{r}{run_equilibrium_sa-C} & \multicolumn{3}{r}{run_equilibrium_sa-M} \\
 & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & N

...

\begin{table}
\caption{Iteration counts (households)}
\label{tab:val-iters-households}
\begin{tabular}{lllllllllllllllllllllllllllllllllllll}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_dp_solve-C} & \multicolumn{3}{r}{run_equilibrium_dp_solve-M} & \multicolumn{3}{r}{run_equilibrium_full_ad-C} & \multicolumn{3}{r}{run_equilibrium_full_ad-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} & \multicolumn{3}{r}{run_equilibrium_man_sa-C} & \multicolumn{3}{r}{run_equilibrium_man_sa-M} & \multicolumn{3}{r}{run_equilibrium_sa-C} & \multicolumn{3}{r}{run_equilibrium_sa-M} \\
 & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT \\
n &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\

label run_equilibrium-C             run_equilibrium-M              \
                 Newton  Conv Match            Newton  Conv Match   
abar                                                                
10                    5  True  True                 5  True  True   
15                    5  True  True                 5  True  True   
20                    5  True  True                 5  True  True   
25                    6  True  True                 6  True  True   
30                    6  True  True                 6  True  True   
35                    6  True  True                 6  True  True   

label run_equilibrium_dp_solve-C             run_equilibrium_dp_solve-M        \
                          Newton  Conv Match                     Newton  Conv   
abar                                                                            
10                             5  True  True                          5  True   
15                             5  True  True                          5  True   
20                             5  True  True                          5  True   
25                             6  True  True                          6  True   
30                             6  True  True                          6  True   
35                             6  True  True                          6  True   

label       run_equilibrium_full_ad-C             run_equilibrium_full_ad-M  \
      Match                    Newton  Conv Match                    Newton   
abar                                                                          
10     True                         6  True  True                         6   
15     True                         6  True  True                         6   
20     True                         6  True  True                         6   
25     True                         7  True  True                         7   
30     True                         7  True  True                         7   
35     True                         7  True  True                         7   

label             run_equilibrium_man-C             run_equilibrium_man-M  \
       Conv Match                Newton  Conv Match                Newton   
abar                                                                        
10     True  True                     5  True  True                     5   
15     True  True                     5  True  True                     5   
20     True  True                     5  True  True                     5   
25     True  True                     6  True  True                     6   
30     True  True                     6  True  True                     6   
35     True  True                     6  True  True                     6   

label             run_equilibrium_man_sa-C                \
       Conv Match                   Newton   Conv  Match   
abar                                                       
10     True  True                       20  False  False   
15     True  True                       20  False  False   
20     True  True                       20  False  False   
25     True  True                       20  False  False   
30     True  True                       16   True  False   
35     True  True                       13   True  False   

label run_equilibrium_man_sa-M               run_equilibrium_sa-C              \
                        Newton   Conv  Match               Newton  Conv Match   
abar                                                                            
10                          20  False  False                    6  True  True   
15                          20  False  False                    6  True  True   
20                          20  False  False                    6  True  True   
25                          20  False  False                    7  True  True   
30                          16   True  False                    7  True  True   
35                          13   True  False         

\begin{table}
\caption{Convergence summary (age)}
\label{tab:val-summary-age}
\begin{tabular}{lrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrrr}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_dp_solve-C} & \multicolumn{3}{r}{run_equilibrium_dp_solve-M} & \multicolumn{3}{r}{run_equilibrium_full_ad-C} & \multicolumn{3}{r}{run_equilibrium_full_ad-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} & \multicolumn{3}{r}{run_equilibrium_man_sa-C} & \multicolumn{3}{r}{run_equilibrium_man_sa-M} & \multicolumn{3}{r}{run_equilibrium_sa-C} & \multicolumn{3}{r}{run_equilibrium_sa-M} \\
 & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match & Newton & Conv & Match \\
abar &  &  &  &

label run_equilibrium-C       run_equilibrium-M        \
                     SA NK RT                SA NK RT   
abar                                                    
10                   20  5  5                20  5  5   
15                   20  5  5                20  5  5   
20                   20  5  5                20  5  5   
25                   24  6  6                24  6  6   
30                   24  6  6                24  6  6   
35                   24  6  6                24  6  6   

label run_equilibrium_dp_solve-C         run_equilibrium_dp_solve-M          \
                              SA  NK  RT                         SA  NK  RT   
abar                                                                          
10                            -1  -1  -1                         -1  -1  -1   
15                            -1  -1  -1                         -1  -1  -1   
20                            -1  -1  -1                         -1  -1  -1   
25                            -1  -1  -1                         -1  -1  -1   
30                            -1  -1  -1                         -1  -1  -1   
35                            -1  -1  -1                         -1  -1  -1   

label run_equilibrium_full_ad-C       run_equilibrium_full_ad-M        \
                             SA NK RT                        SA NK RT   
abar                                                                    
10                           24  6  6                        24  6  6   
15                           24  6  6                        24  6  6   
20                           24  6  6                        24  6  6   
25                           28  7  7                        28  7  7   
30                           28  7  7                        28  7  7   
35                           28  7  7                        28  7  7   

label run_equilibrium_man-C       run_equilibrium_man-M        \
                         SA NK RT                    SA NK RT   
abar                                                            
10                       20  5  5                    20  5  5   
15                       20  5  5                    20  5  5   
20                       20  5  5                    20  5  5   
25                       24  6  6                    24  6  6   
30                       24  6  6                    24  6  6   
35                       24  6  6                    24  6  6   

label run_equilibrium_man_sa-C       run_equilibrium_man_sa-M        \
                            SA NK RT                       SA NK RT   
abar                                                                  
10                         400  0  0                      400  0  0   
15                         400  0  0                      400  0  0   
20                         400  0  0                      400  0  0   
25                         400  0  0                      400  0  0   
30                         320  0  0                      320  0  0   
35                         260  0  0                      260  0  0   

label run_equilibrium_sa-C       run_equilibrium_sa-M        
                        SA NK RT                   SA NK RT  
abar                                                         
10               2909,2666  0  0            2909,2666  0  0  
15               2917,2785  0  0            2917,2785  0  0  
20               2917,2821  0  0            2917,2821  0  0  
25               3403,3295  0  0            3403,3295  0  0  
30               3403,3295  0  0            3403,3295  0  0  
35               3403,3295  0  0            3403,3295  0  0

\begin{table}
\caption{Iteration counts (age)}
\label{tab:val-iters-age}
\begin{tabular}{lllllllllllllllllllllllllllllllllllll}
\toprule
label & \multicolumn{3}{r}{run_equilibrium-C} & \multicolumn{3}{r}{run_equilibrium-M} & \multicolumn{3}{r}{run_equilibrium_dp_solve-C} & \multicolumn{3}{r}{run_equilibrium_dp_solve-M} & \multicolumn{3}{r}{run_equilibrium_full_ad-C} & \multicolumn{3}{r}{run_equilibrium_full_ad-M} & \multicolumn{3}{r}{run_equilibrium_man-C} & \multicolumn{3}{r}{run_equilibrium_man-M} & \multicolumn{3}{r}{run_equilibrium_man_sa-C} & \multicolumn{3}{r}{run_equilibrium_man_sa-M} & \multicolumn{3}{r}{run_equilibrium_sa-C} & \multicolumn{3}{r}{run_equilibrium_sa-M} \\
 & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT & SA & NK & RT \\
abar &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
10 

In [103]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

display(comparison_table("cars"))

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

,run_equi,val_name,n,c,abar,acc0,trans,backend,filename,max_abs_ed,stat_res,norm_err,min_q,iter,conv,sa_iters_tot,nk_iters_tot,rtrips_tot,within_tol,max_abs_diff,max_rel_diff
0,run_equilibrium,3_local_new,2,1,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-1...,1.776060e-10,3.000000e-15,0.000000e+00,4.725005e-04,7,True,"[30, 33]","[12, 7]","[7, 7]",True,0.000002,1.041376e-07
1,run_equilibrium,3_local_new,2,3,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-3...,1.500000e-14,5.300000e-14,0.000000e+00,2.429617e-04,8,True,"[32, 32]","[8, 8]","[8, 8]",True,0.000004,1.298640e-07
2,run_equilibrium,3_local_new,2,5,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-5...,1.199000e-12,3.700000e-14,1.000000e-15,1.645633e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,2.444766e-07
3,run_equilibrium,3_local_new,2,7,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-7...,9.000000e-14,2.500000e-13,0.000000e+00,1.245906e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,1.635894e-07
4,run_equilibrium,3_local_new,2,9,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-9...,1.601310e-10,1.190000e-13,1.000000e-15,1.002891e-04,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.857835e-07
5,run_equilibrium,3_local_new,2,11,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-1...,1.488700e-11,3.640000e-13,1.000000e-15,8.393742e-05,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.817334e-07
6,run_equilibrium,3_local_new,2,13,25,5,0,C,run_equilibrium-3_local_new-validate_solve-2-1...,3.034000e-12,2.380000e-13,1.000000e-15,7.217770e-05,6,True,"[24, 24]","[6, 6]","[6, 6]",True,0.000004,1.614682e-07
7,run_equilibrium,3_local_new,2,1,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-1...,1.776060e-10,6.000000e-15,0.000000e+00,4.725005e-04,7,True,"[30, 33]","[12, 7]","[7, 7]",True,0.000002,1.041376e-07
8,run_equilibrium,3_local_new,2,3,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-3...,1.500000e-14,5.300000e-14,0.000000e+00,2.429617e-04,8,True,"[32, 32]","[8, 8]","[8, 8]",True,0.000004,1.298640e-07
9,run_equilibrium,3_local_new,2,5,25,5,0,M,run_equilibrium-3_local_new-validate_solve-2-5...,1.199000e-12,3.700000e-14,1.000000e-15,1.645633e-04,7,True,"[28, 28]","[7, 7]","[7, 7]",True,0.000004,2.444766e-07


In [ ]:
SYSTEMS = ["local", "futhark01", "futhark03"]
RUN_EQUIS = ["run_equilibrium", "run_equilibrium_man", "run_equilibrium_full_ad"]
BACKEND_ORDER = {"C": 0, "M": 1, "U": 2, "O": 3}

def _iter_range(x):
    x = pd.Series(x).dropna().astype(int)
    if x.empty:
        return ""
    lo, hi = x.min(), x.max()
    return str(lo) if lo == hi else f"{lo}--{hi}"


def validation_summary_selected(variant="cars"):
    """
    Summarise validation results for local/futhark01/futhark03 and
    run_equilibrium / run_equilibrium_man / run_equilibrium_full_ad.

    For car scaling, use variant='cars'. This includes the extended c values
    such as 17 and 21, since cars_long is only separate for benchmark files.
    """
    df = comparison_table(variant)
    df = df[
        df["val_name"].isin(SYSTEMS)
        & df["run_equi"].isin(RUN_EQUIS)
    ].copy()

    out = (
        df.groupby(["val_name", "run_equi", "backend"], as_index=False)
          .agg(
              configs=("filename", "count"),
              all_converged=("conv", "all"),
              max_abs_diff=("max_abs_diff", "max"),
              max_abs_ed=("max_abs_ed", "max"),
              max_stat_res=("stat_res", "max"),
              max_norm_err=("norm_err", "max"),
              min_q=("min_q", "min"),
              newton_iters=("iter", _iter_range),
          )
    )

    out["val_name"] = pd.Categorical(out["val_name"], SYSTEMS, ordered=True)
    out["run_equi"] = pd.Categorical(out["run_equi"], RUN_EQUIS, ordered=True)
    out["backend_order"] = out["backend"].map(BACKEND_ORDER).fillna(99)

    return (
        out.sort_values(["val_name", "run_equi", "backend_order"])
           .drop(columns=["backend_order"])
           .reset_index(drop=True)
    )


def validation_details_selected(variant="cars"):
    """
    Detailed validation rows for local/futhark01/futhark03 and
    run_equilibrium / run_equilibrium_man / run_equilibrium_full_ad.
    """
    df = comparison_table(variant)
    df = df[
        df["val_name"].isin(SYSTEMS)
        & df["run_equi"].isin(RUN_EQUIS)
    ].copy()

    sort_col = {"cars": "c", "households": "n", "age": "abar"}[variant]
    df["val_name"] = pd.Categorical(df["val_name"], SYSTEMS, ordered=True)
    df["run_equi"] = pd.Categorical(df["run_equi"], RUN_EQUIS, ordered=True)
    df["backend_order"] = df["backend"].map(BACKEND_ORDER).fillna(99)

    cols = [
        "val_name", "run_equi", "backend",
        "n", "c", "abar", "acc0", "trans",
        "conv", "iter",
        "max_abs_diff", "max_abs_ed", "stat_res", "norm_err", "min_q",
        "sa_iters_tot", "nk_iters_tot", "rtrips_tot",
        "filename",
    ]

    return (
        df.sort_values(["val_name", "run_equi", "backend_order", sort_col])
          .drop(columns=["backend_order"])
          [cols]
          .reset_index(drop=True)
    )


def benchmark_selected(variant="cars_long"):
    """
    Benchmark rows for local/futhark01/futhark03 and
    run_equilibrium / run_equilibrium_man / run_equilibrium_full_ad.

    Use variant='cars_long', 'cars_longer', or 'cars_longest'
    for the car-scaling benchmarks.
    """
    df = bench_time_table(variant)

    mask = (
        (df["source"] == "matlab")
        | (
            (df["source"] == "futhark")
            & df["val_name"].isin(SYSTEMS)
            & df["run_equi"].isin(RUN_EQUIS)
        )
    )

    df = df[mask].copy()

    sort_col = {
        "cars": "c",
        "cars_long": "c",
        "cars_longer": "c",
        "cars_longest": "c",
        "households": "n",
        "age": "abar",
    }[variant]

    df["val_name"] = pd.Categorical(df["val_name"], ["-", *SYSTEMS], ordered=True)
    df["run_equi"] = pd.Categorical(df["run_equi"], ["-", *RUN_EQUIS], ordered=True)
    df["backend_order"] = df["backend"].map(BACKEND_ORDER).fillna(99)

    return (
        df.sort_values([sort_col, "source", "val_name", "run_equi", "backend_order"])
          .drop(columns=["backend_order"])
          .reset_index(drop=True)
    )


pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

print("=== Validation summary: selected systems and solvers ===")
display(validation_summary_selected("cars"))

print("=== Validation details: selected systems and solvers ===")
display(validation_details_selected("cars"))

for variant in ["cars_long", "cars_longer", "cars_longest"]:
    print(f"=== Benchmark results: selected systems and solvers ({variant}) ===")
    display(benchmark_selected(variant))

In [105]:
CAR_VARIANT_PRIORITY = {
    "cars_long": 0,
    "cars_longer": 1,
    "cars_longest": 2,
}

def combined_car_benchmarks_prefer_latest():
    dfs = []
    for variant in ["cars_long", "cars_longer", "cars_longest"]:
        df = bench_time_table(variant).copy()
        if not df.empty:
            df["variant"] = variant
            df["variant_priority"] = CAR_VARIANT_PRIORITY[variant]
            dfs.append(df)

    if not dfs:
        return pd.DataFrame()

    df = pd.concat(dfs, ignore_index=True)

    # For the same configuration, keep the result from the latest / most precise variant.
    key_cols = ["source", "run_equi", "val_name", "backend", "n", "c", "abar"]
    df = (
        df.sort_values("variant_priority")
          .drop_duplicates(subset=key_cols, keep="last")
          .drop(columns=["variant_priority"])
          .reset_index(drop=True)
    )

    return df


def benchcell_tex(runtime, speedup=None):
    if pd.isna(runtime):
        return ""
    if speedup is None or pd.isna(speedup):
        return f"{runtime:.3f}"
    return rf"\benchcell{{{runtime:.3f}}}{{{speedup:.1f}}}"


def make_futhark01_common_table():
    df = combined_car_benchmarks_prefer_latest()
    df = df[df["c"].isin([1, 5, 9, 13, 17, 21])].copy()

    df = df[
        (df["source"].eq("matlab"))
        |
        (
            df["source"].eq("futhark")
            & df["val_name"].eq("futhark01")
            & df["run_equi"].isin(["run_equilibrium", "run_equilibrium_man"])
            & df["backend"].isin(["C", "M", "U"])
        )
    ].copy()

    matlab = df[df["source"].eq("matlab")].set_index("c")["mean"].rename("MATLAB")

    futhark = df[df["source"].eq("futhark")].copy()
    futhark["label"] = futhark.apply(
        lambda r:
            "Base-C" if r["run_equi"] == "run_equilibrium" and r["backend"] == "C" else
            "Base-M" if r["run_equi"] == "run_equilibrium" and r["backend"] == "M" else
            "Base-CUDA" if r["run_equi"] == "run_equilibrium" and r["backend"] == "U" else
            "Manual-C" if r["run_equi"] == "run_equilibrium_man" and r["backend"] == "C" else
            "Manual-M" if r["run_equi"] == "run_equilibrium_man" and r["backend"] == "M" else
            "Manual-CUDA" if r["run_equi"] == "run_equilibrium_man" and r["backend"] == "U" else
            None,
        axis=1,
    )

    wide = futhark.pivot_table(index="c", columns="label", values="mean", aggfunc="first")
    wide = wide.join(matlab, how="outer")
    wide = wide.reindex(columns=["MATLAB", "Base-C", "Base-M", "Base-CUDA", "Manual-C", "Manual-M", "Manual-CUDA"])
    wide = wide.sort_index()

    lines = [
        r"\begin{table}[H]",
        r"\caption{\texttt{futhark01} car-type benchmark}",
        r"\label{tab:bench-cars-long-futhark01}",
        r"\centering",
        r"\scriptsize",
        r"\renewcommand{\arraystretch}{1.15}",
        r"\begin{tabular}{rccccccc}",
        r"\hline",
        r"Car types & MATLAB & Base-C & Base-M & Base-CUDA & Manual-C & Manual-M & Manual-CUDA \\",
        r"\hline",
    ]

    for c, row in wide.iterrows():
        matlab_time = row["MATLAB"]
        cells = [str(int(c)), f"{matlab_time:.3f}" if pd.notna(matlab_time) else ""]
        for col in ["Base-C", "Base-M", "Base-CUDA", "Manual-C", "Manual-M", "Manual-CUDA"]:
            runtime = row[col]
            speedup = matlab_time / runtime if pd.notna(matlab_time) and pd.notna(runtime) else None
            cells.append(benchcell_tex(runtime, speedup))
        lines.append(" & ".join(cells) + r" \\")

    lines += [
        r"\hline",
        r"\end{tabular}",
        "",
        r"\begin{flushleft}",
        r"\footnotesize",
        r"\textit{Note:} Mean equilibrium-solver runtimes in seconds for \texttt{futhark01}. Speedup relative to MATLAB on \texttt{futhark01} for the same number of car types is shown below each runtime in parentheses. MATLAB runtimes are averaged over 10 benchmark runs. Futhark C-backend runtimes are averaged over 3 runs, while Futhark multicore and CUDA runtimes are averaged over 6 runs where available. The benchmark uses two household types and maximal age 25. Base denotes the AD-based implementation, while Manual denotes the implementation using manually derived derivatives.",
        r"\end{flushleft}",
        r"\end{table}",
    ]

    return "\n".join(lines)


def make_futhark01_extended_table():
    df = combined_car_benchmarks_prefer_latest()
    df = df[df["c"] > 21].copy()

    df = df[
        df["source"].eq("futhark")
        & df["val_name"].eq("futhark01")
        & df["run_equi"].isin(["run_equilibrium", "run_equilibrium_man"])
        & df["backend"].isin(["M", "U"])
    ].copy()

    futhark = df.copy()
    futhark["label"] = futhark.apply(
        lambda r:
            "Base-M" if r["run_equi"] == "run_equilibrium" and r["backend"] == "M" else
            "Base-CUDA" if r["run_equi"] == "run_equilibrium" and r["backend"] == "U" else
            "Manual-M" if r["run_equi"] == "run_equilibrium_man" and r["backend"] == "M" else
            "Manual-CUDA" if r["run_equi"] == "run_equilibrium_man" and r["backend"] == "U" else
            None,
        axis=1,
    )

    wide = futhark.pivot_table(index="c", columns="label", values="mean", aggfunc="first")
    wide = wide.reindex(columns=["Base-M", "Base-CUDA", "Manual-M", "Manual-CUDA"])
    wide = wide.sort_index()

    lines = [
        r"\begin{table}[H]",
        r"\caption{Extended \texttt{futhark01} car-type benchmark}",
        r"\label{tab:bench-cars-extended-futhark01}",
        r"\centering",
        r"\small",
        r"\renewcommand{\arraystretch}{1.15}",
        r"\begin{tabular}{rcccc}",
        r"\hline",
        r"Car types & Base-M & Base-CUDA & Manual-M & Manual-CUDA \\",
        r"\hline",
    ]

    for c, row in wide.iterrows():
        cells = [str(int(c))]
        for col in ["Base-M", "Base-CUDA", "Manual-M", "Manual-CUDA"]:
            cells.append(f"{row[col]:.3f}" if pd.notna(row[col]) else "")
        lines.append(" & ".join(cells) + r" \\")

    lines += [
        r"\hline",
        r"\end{tabular}",
        "",
        r"\begin{flushleft}",
        r"\footnotesize",
        r"\textit{Note:} Mean equilibrium-solver runtimes in seconds for the extended \texttt{futhark01} benchmark. These runs include only the multicore and CUDA backends for car-type configurations larger than those included in Table~\ref{tab:bench-cars-long-futhark01}. Each reported runtime is averaged over 6 benchmark runs. The benchmark uses two household types and maximal age 25. Base denotes the AD-based implementation, while Manual denotes the implementation using manually derived derivatives.",
        r"\end{flushleft}",
        r"\end{table}",
    ]

    return "\n".join(lines)


print(make_futhark01_common_table())
print("\n\n")
print(make_futhark01_extended_table())

\begin{table}[H]
\caption{\texttt{futhark01} car-type benchmark}
\label{tab:bench-cars-long-futhark01}
\centering
\scriptsize
\renewcommand{\arraystretch}{1.15}
\begin{tabular}{rccccccc}
\hline
Car types & MATLAB & Base-C & Base-M & Base-CUDA & Manual-C & Manual-M & Manual-CUDA \\
\hline
1 & 0.106 & \benchcell{0.077}{1.4} & \benchcell{0.058}{1.8} & \benchcell{0.283}{0.4} & \benchcell{0.024}{4.3} & \benchcell{0.066}{1.6} & \benchcell{0.286}{0.4} \\
5 & 1.754 & \benchcell{3.935}{0.4} & \benchcell{0.483}{3.6} & \benchcell{1.326}{1.3} & \benchcell{0.781}{2.2} & \benchcell{0.377}{4.7} & \benchcell{1.300}{1.3} \\
9 & 10.212 & \benchcell{17.634}{0.6} & \benchcell{1.453}{7.0} & \benchcell{3.127}{3.3} & \benchcell{3.224}{3.2} & \benchcell{1.172}{8.7} & \benchcell{2.403}{4.2} \\
13 & 33.853 & \benchcell{54.497}{0.6} & \benchcell{3.178}{10.7} & \benchcell{5.814}{5.8} & \benchcell{9.979}{3.4} & \benchcell{3.313}{10.2} & \benchcell{4.976}{6.8} \\
17 & 118.626 & \benchcell{126.821}{0.9} & \benchcell

##### CUDA vs non-CUDA prices (configs without a MATLAB reference)

In [106]:
def cuda_vs_noncuda_diffs(val_dir='validation_files',
                          mat_dir='matlab_results_for_validation'):
    """Per-config max |U_prices - other_backend_prices| where both a CUDA ('U')
    .val file and at least one non-CUDA .val file exist, and there is no
    matching MATLAB reference file.

    Returns a DataFrame with one row per (config, non-CUDA backend).
    """
    val_dir = Path(val_dir)
    mat_dir = Path(mat_dir)

    groups = {}
    for path in sorted(val_dir.glob('*.val')):
        m = _FUT_FILE_RE.match(path.name)
        if m is None:
            continue
        key = (m['run_equi'], m['val_name'],
               int(m['n']), int(m['c']), int(m['abar']),
               int(m['acc0']), int(m['trans']))
        groups.setdefault(key, {})[m['backend']] = path

    results = []
    for key, backends in groups.items():
        run_equi, val_name, n, c, abar, acc0, trans = key

        mat_path = mat_dir / f"matlab-validate_solve-{n}-{c}-{abar}-{acc0}-{trans}.dat"
        if mat_path.exists():
            continue
        if 'U' not in backends or len(backends) < 2:
            continue

        u_prices = parse_futhark_prices(backends['U'])
        for b, p in backends.items():
            if b == 'U':
                continue
            other = parse_futhark_prices(p)
            diff = np.abs(u_prices - other)
            results.append({
                'run_equi': run_equi, 'val_name': val_name,
                'n': n, 'c': c, 'abar': abar, 'acc0': acc0, 'trans': trans,
                'other_backend': b,
                'max_abs_diff': float(np.max(diff)),
                'max_rel_diff': float(np.max(diff / np.maximum(np.abs(other), 1e-12))),
            })

    df = pd.DataFrame(results)
    if not df.empty:
        df = df.sort_values(['run_equi', 'val_name', 'n', 'c', 'abar',
                             'acc0', 'trans', 'other_backend']
                           ).reset_index(drop=True)
    return df


display(cuda_vs_noncuda_diffs())

,run_equi,val_name,n,c,abar,acc0,trans,other_backend,max_abs_diff,max_rel_diff
0,run_equilibrium,futhark01,2,25,25,5,0,M,0.003621,0.000193
1,run_equilibrium,futhark01,2,29,25,5,0,M,0.002368,0.000127
2,run_equilibrium,futhark01,2,33,25,5,0,M,0.000134,0.000009
3,run_equilibrium_almost_full_ad,futhark01,2,25,25,5,0,M,0.003621,0.000193
4,run_equilibrium_almost_full_ad,futhark01,2,29,25,5,0,M,0.002368,0.000127
5,run_equilibrium_almost_full_ad,futhark01,2,33,25,5,0,M,0.000134,0.000009
6,run_equilibrium_man,futhark01,2,25,25,5,0,M,0.003621,0.000193
7,run_equilibrium_man,futhark01,2,29,25,5,0,M,0.002368,0.000127
8,run_equilibrium_man,futhark01,2,33,25,5,0,M,0.000134,0.000009
9,run_equilibrium_man,futhark01,2,40,25,5,0,M,0.000079,0.000005
